# 第73章 用户行为数据分析

餐厅想提高小费收入，但不知道该从哪里下手：是抓大单、调班次、还是改服务对象？本项目用真实的 244 条餐厅账单记录，做分层归因和统计推断，区分「看起来有差异」和「真的有差异」。

## 项目背景

数据来源：plotly 内置 tips 数据集，原始出自 Bryant & Smith《Practical Data Analysis: Case Studies in Business Statistics》(1995)，记录了美国某餐厅一名服务员连续若干天的 244 笔真实账单。字段包含账单金额、小费、顾客性别、是否吸烟区、星期、午/晚餐、同行人数。\n\n样本量只有 244——这恰恰是真实业务分析的常态。小样本下最容易犯的错误是把随机波动当成规律，所以本项目的重点不是画图，而是**证明差异存在**。

## 学习目标

- 掌握派生指标的设计：为什么要用小费率而不是小费绝对值做分析
- 掌握分层分析（stratification）：逐维度拆解 + 样本量校验
- 识别辛普森悖论：交叉分层后结论反转的成因
- 在不依赖 scipy 的前提下，用置换检验判断组间差异的统计显著性
- 用自举法（bootstrap）给出效应量的置信区间，而不是只报一个点估计


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| total_bill | 账单总额（美元） | 不含小费 |
| tip | 小费金额（美元） | 被解释变量的原始形态 |
| sex | 付款人性别 | Male / Female |
| smoker | 是否吸烟区 | Yes / No |
| day | 星期 | Thur / Fri / Sat / Sun |
| time | 餐段 | Lunch / Dinner |
| size | 同行人数 | 1-6 |
| tip_pct | 小费率（派生） | = tip / total_bill，本项目核心指标 |

## 数据质量检查清单

- 确认无缺失值与非法值（账单金额、小费必须为正）
- 检查重复行：完全相同的账单是否真实存在
- 逐维度统计各分组样本量，标记 n < 20 的格子为「不可结论区」
- 检查 day 与 time 的交叉分布，确认存在结构性空缺（周末无午餐记录）


## 项目任务

1. 构造小费率指标，并说明为什么它比小费绝对值更适合做跨账单比较
2. 对性别、吸烟区、星期、餐段、人数逐一做单因素分层，记录均值差与样本量
3. 做「餐段 × 星期」交叉表，找出单因素结论在交叉后被推翻的维度
4. 对最大的那个组间差异做置换检验，报告 p 值
5. 用自举法给出该差异的 95% 置信区间，判断它是否跨过 0
6. 汇总证据，给出可执行的行动建议并标注置信度


## 步骤1：载入真实账单数据并构造核心指标

小费绝对值受账单规模主导，跨账单不可比。分析的第一个决策是把被解释变量换成**比率**：小费率 = 小费 / 账单额。这一步不是数据清洗，而是分析框架的设定——选错指标，后面所有统计都在回答错误的问题。


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# 真实公开数据：Bryant & Smith (1995) 餐厅小费数据，随 plotly 离线分发
tips = px.data.tips()

print("=" * 92)
print("数据集：餐厅账单与小费（244 笔真实交易）")
print("=" * 92)
print(f"形状: {tips.shape[0]} 行 × {tips.shape[1]} 列")
print(f"字段: {list(tips.columns)}")
print()
print(tips.head(8).to_string(index=False))

# 核心派生指标
tips["tip_pct"] = tips["tip"] / tips["total_bill"]
tips["bill_per_person"] = tips["total_bill"] / tips["size"]

print("\n" + "-" * 92)
print("为什么用小费率而不是小费金额？")
print("-" * 92)
print(f"小费金额 与 账单额 的相关系数 : {tips['tip'].corr(tips['total_bill']):.3f}  ← 强相关，被账单规模主导")
print(f"小费率   与 账单额 的相关系数 : {tips['tip_pct'].corr(tips['total_bill']):.3f}  ← 已剥离规模效应")
print()
print("结论：小费金额高，可能只是账单大。要衡量「顾客大方程度」，必须用比率。")

print("\n" + "-" * 92)
print("核心指标分布")
print("-" * 92)
desc = tips["tip_pct"].describe()
print(f"样本量 : {desc['count']:.0f}")
print(f"均值   : {desc['mean']:.4f}  ({desc['mean'] * 100:.2f}%)")
print(f"中位数 : {desc['50%']:.4f}  ({desc['50%'] * 100:.2f}%)")
print(f"标准差 : {desc['std']:.4f}")
print(f"范围   : {desc['min']:.4f} ~ {desc['max']:.4f}")
print()
print(f"均值 > 中位数，说明分布右偏：少数高小费率账单拉高了均值。")
print(f"→ 后续做组间比较时，均值和中位数都要看。")


## 步骤2：数据质量审计与「不可结论区」标记

244 行的数据集拆到「餐段 × 星期」这一层，某些格子可能只剩几条记录。分析者的职业素养体现在：**先把不能下结论的地方标出来**，而不是等画完图再解释为什么某根柱子是异常的。这里同时会发现一个结构性空缺——周末没有午餐记录。


In [ ]:
print("=" * 92)
print("数据质量审计")
print("=" * 92)

# 1. 缺失与非法值
print("\n[1] 缺失值与合法性")
missing = tips.isna().sum()
print(f"    缺失值总数: {int(missing.sum())}")
illegal = ((tips["total_bill"] <= 0) | (tips["tip"] < 0)).sum()
print(f"    非法值（账单<=0 或 小费<0）: {int(illegal)}")
extreme = (tips["tip_pct"] > 0.5).sum()
print(f"    小费率 > 50% 的记录: {int(extreme)} 条  ← 保留，真实存在的慷慨顾客")

# 2. 重复行
print("\n[2] 重复行检查")
dup = tips.duplicated().sum()
print(f"    完全重复的行: {int(dup)} 条")
if dup > 0:
    print("    判断：两桌顾客账单、小费、人数完全相同是可能的，属于真实巧合，不删除。")

# 3. 分组样本量 —— 决定哪些结论可以下
print("\n[3] 各维度分组样本量（n < 20 视为不可结论区）")
for col in ["sex", "smoker", "day", "time", "size"]:
    counts = tips[col].value_counts().sort_index()
    parts = []
    for k, v in counts.items():
        flag = "!" if v < 20 else " "
        parts.append(f"{k}={v}{flag}")
    print(f"    {col:8s}: {'  '.join(parts)}")
print("    （标记 ! 的分组样本过少，其均值波动大，不单独下结论）")

# 4. 交叉分布 —— 发现结构性空缺
print("\n[4] 餐段 × 星期 交叉样本量")
cross = pd.crosstab(tips["day"], tips["time"])
print(cross.to_string())

zeros = [(d, t) for d in cross.index for t in cross.columns if cross.loc[d, t] == 0]
print(f"\n    空格子: {zeros}")
print("    这不是数据缺失，是业务事实：该餐厅周末不营业午市。")
print("    → 任何「午餐 vs 晚餐」的比较，实际上混入了「工作日 vs 周末」的差异。")
print("    → 这就是下一步必须做交叉分层的原因。")


## 步骤3：单因素分层 —— 逐维度找差异

分层分析是归因的起点：把总体按每个维度切开，看指标在哪个维度上分化最明显。关键是**同时输出效应量和样本量**——一个 5 个百分点的差异，如果只建立在 19 条记录上，那它更可能是噪声。这一步只做筛选，不下结论。


In [ ]:
print("=" * 92)
print("单因素分层分析：小费率在哪个维度上分化最明显")
print("=" * 92)

def stratify(df, col, metric="tip_pct"):
    g = df.groupby(col, observed=True)[metric].agg(["count", "mean", "median", "std"])
    g["mean_pct"] = g["mean"] * 100
    return g.sort_values("mean", ascending=False)

summary_rows = []
for col in ["sex", "smoker", "time", "day", "size"]:
    g = stratify(tips, col)
    print(f"\n--- 按 {col} 分层 ---")
    for idx, row in g.iterrows():
        bar = "#" * int(row["mean"] * 200)
        warn = "  <- 样本过少" if row["count"] < 20 else ""
        print(f"  {str(idx):8s} n={int(row['count']):3d}  均值={row['mean_pct']:5.2f}%  "
              f"中位数={row['median'] * 100:5.2f}%  {bar}{warn}")

    # 记录该维度的最大组间差（仅统计样本量达标的组）
    valid = g[g["count"] >= 20]
    if len(valid) >= 2:
        spread = valid["mean"].max() - valid["mean"].min()
        summary_rows.append({
            "维度": col,
            "有效分组数": len(valid),
            "最高组": str(valid["mean"].idxmax()),
            "最低组": str(valid["mean"].idxmin()),
            "组间差(百分点)": spread * 100,
            "最小组样本": int(valid["count"].min()),
        })

rank = pd.DataFrame(summary_rows).sort_values("组间差(百分点)", ascending=False)
print("\n" + "=" * 92)
print("维度重要性排序（按最大组间差，仅含样本量>=20 的组）")
print("=" * 92)
print(rank.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

top = rank.iloc[0]
print(f"\n候选主因：{top['维度']}（{top['最高组']} vs {top['最低组']}，"
      f"相差 {top['组间差(百分点)']:.2f} 个百分点）")
print("注意：这只是描述性差异。是否显著，要到步骤5做检验。")


## 步骤4：交叉分层 —— 主动搜索辛普森悖论

单因素分层的致命缺陷是混淆变量。步骤2 已经发现「午餐」只出现在工作日——所以「午餐 vs 晚餐」的差异里混着「工作日 vs 周末」。这一步不靠直觉猜，而是**程序化枚举所有 二元因素 × 控制变量 的组合，自动标记符号反转**。凡是总体方向和分层后方向不一致的，都是结论不可靠的信号。


In [ ]:
print("=" * 92)
print("交叉分层：搜索符号反转（辛普森悖论）")
print("=" * 92)

def mean_diff(df, factor, high, low, metric="tip_pct"):
    """返回 (均值差, high组样本量, low组样本量)"""
    a = df.loc[df[factor] == high, metric]
    b = df.loc[df[factor] == low, metric]
    if len(a) == 0 or len(b) == 0:
        return np.nan, len(a), len(b)
    return a.mean() - b.mean(), len(a), len(b)

binary_factors = {"sex": ("Male", "Female"), "smoker": ("Yes", "No"), "time": ("Lunch", "Dinner")}
controls = ["day", "time", "size", "smoker", "sex"]
MIN_N = 15

reversals = []
for factor, (hi, lo) in binary_factors.items():
    overall, n_hi, n_lo = mean_diff(tips, factor, hi, lo)
    print(f"\n--- 因素 {factor}: {hi} vs {lo} ---")
    print(f"  总体差异: {overall * 100:+.2f} 个百分点  (n={n_hi} vs {n_lo})")

    for ctrl in controls:
        if ctrl == factor:
            continue
        print(f"  控制 {ctrl} 后：")
        for level in sorted(tips[ctrl].unique(), key=str):
            sub = tips[tips[ctrl] == level]
            d, na, nb = mean_diff(sub, factor, hi, lo)
            if np.isnan(d):
                print(f"    {ctrl}={level!s:8s} 该层缺少对照组，无法比较")
                continue
            reliable = na >= MIN_N and nb >= MIN_N
            mark = ""
            if reliable and np.sign(d) != np.sign(overall):
                mark = "  <== 符号反转"
                reversals.append((factor, ctrl, str(level), overall * 100, d * 100, na, nb))
            note = "" if reliable else "  (样本不足，仅供参考)"
            print(f"    {ctrl}={level!s:8s} 差异={d * 100:+6.2f} pp  (n={na:3d} vs {nb:3d}){mark}{note}")

print("\n" + "=" * 92)
print("搜索结果")
print("=" * 92)
if reversals:
    rev = pd.DataFrame(reversals, columns=["因素", "控制变量", "层级", "总体差异pp", "该层差异pp", "n_high", "n_low"])
    print(rev.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
    print("\n以上组合中，总体结论与分层结论方向相反 —— 这就是辛普森悖论的现场。")
    print("成因：因素与控制变量不独立，分组样本占比不均，聚合时被权重扭曲。")
else:
    print("在样本量达标的层级中未发现符号反转，主效应方向稳健。")

print("\n结构性提醒：")
lunch_days = sorted(tips.loc[tips["time"] == "Lunch", "day"].unique())
dinner_days = sorted(tips.loc[tips["time"] == "Dinner", "day"].unique())
print(f"  午餐仅出现在 {lunch_days}")
print(f"  晚餐出现在   {dinner_days}")
print("  → 「午餐 vs 晚餐」无法与「工作日 vs 周末」分离，此维度的因果解读必须放弃。")


## 步骤5：置换检验 —— 差异是真的还是随机的

小样本下最容易犯的错是把噪声当规律。置换检验的逻辑极其直观：假设分组标签毫无意义，那么把标签随机打乱几千次，得到的差异分布就是「纯随机能达到的水平」；如果真实差异落在这个分布的极端尾部，才说明分组有意义。它不需要正态假设，也不需要 scipy——只用 numpy 就能实现，而且比 t 检验更容易讲清楚原理。


In [ ]:
print("=" * 92)
print("置换检验：组间差异的统计显著性")
print("=" * 92)

def permutation_test(a, b, n_iter=5000, seed=42):
    """双尾置换检验。返回 (观测差异, p值, 零分布)"""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    observed = a.mean() - b.mean()
    pooled = np.concatenate([a, b])
    n_a = len(a)
    rng = np.random.default_rng(seed)
    null = np.empty(n_iter)
    for i in range(n_iter):
        shuffled = rng.permutation(pooled)
        null[i] = shuffled[:n_a].mean() - shuffled[n_a:].mean()
    # +1 修正，避免 p=0 这种过度自信的报告
    p_value = (np.sum(np.abs(null) >= abs(observed)) + 1) / (n_iter + 1)
    return observed, p_value, null

def cohens_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    n1, n2 = len(a), len(b)
    s_pool = np.sqrt(((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1)) / (n1 + n2 - 2))
    return (a.mean() - b.mean()) / s_pool if s_pool > 0 else np.nan

def d_label(d):
    ad = abs(d)
    if ad < 0.2:
        return "可忽略"
    if ad < 0.5:
        return "小"
    if ad < 0.8:
        return "中"
    return "大"

candidates = [
    ("性别", "sex", "Male", "Female"),
    ("吸烟区", "smoker", "Yes", "No"),
    ("餐段", "time", "Lunch", "Dinner"),
    ("周末与否", "is_weekend", True, False),
]
tips["is_weekend"] = tips["day"].isin(["Sat", "Sun"])

results = []
for name, col, hi, lo in candidates:
    a = tips.loc[tips[col] == hi, "tip_pct"].values
    b = tips.loc[tips[col] == lo, "tip_pct"].values
    obs, p, null = permutation_test(a, b)
    d = cohens_d(a, b)
    results.append({
        "对比": f"{name}: {hi} vs {lo}",
        "n_high": len(a), "n_low": len(b),
        "差异pp": obs * 100,
        "p值": p,
        "Cohen_d": d,
        "效应量": d_label(d),
        "显著": "是" if p < 0.05 else "否",
    })
    print(f"\n{name}: {hi}({len(a)}) vs {lo}({len(b)})")
    print(f"  观测差异 : {obs * 100:+.2f} 个百分点")
    print(f"  零分布   : 均值={null.mean() * 100:+.3f}pp  标准差={null.std() * 100:.3f}pp")
    print(f"  p 值     : {p:.4f}   {'← 显著 (p<0.05)' if p < 0.05 else '← 不显著，无法排除随机波动'}")
    print(f"  Cohen d  : {d:+.3f} ({d_label(d)}效应)")

res = pd.DataFrame(results)
print("\n" + "=" * 92)
print("检验结果汇总")
print("=" * 92)
print(res.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

sig = res[res["显著"] == "是"]
print(f"\n{len(sig)} / {len(res)} 项对比达到显著水平。")
if len(sig) == 0:
    print("→ 关键结论：步骤3 看到的所有「差异」都无法与随机波动区分。")
    print("  244 个样本不足以支撑按人群细分的运营决策 —— 这是一个有价值的负面结论，")
    print("  它阻止团队基于噪声去调整排班或服务策略。")
else:
    print("→ 仅对显著项做后续决策，其余归入「证据不足」。")


## 步骤6：自举置信区间 —— 把「点估计」换成「区间估计」

只报一个均值是不负责任的：14.9% 这个数字背后可能是 ±0.5pp 的稳定估计，也可能是 ±3pp 的剧烈摇摆，而两者对应完全不同的决策信心。自举（bootstrap）从样本中有放回地重抽同等规模的数据几千次，用重抽结果的分布来量化不确定性。区间宽度会直接暴露哪些分组的样本量根本不够用。


In [ ]:
print("=" * 92)
print("自举置信区间：每个分组的估计精度")
print("=" * 92)

def bootstrap_ci(x, stat=np.mean, n_iter=4000, level=0.95, seed=7):
    """百分位法自举置信区间"""
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) < 2:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = np.array([stat(rng.choice(x, size=len(x), replace=True)) for _ in range(n_iter)])
    alpha = (1 - level) / 2
    return stat(x), np.quantile(boots, alpha), np.quantile(boots, 1 - alpha)

rows = []
overall_pt, overall_lo, overall_hi = bootstrap_ci(tips["tip_pct"])
rows.append(["全体", "-", len(tips), overall_pt, overall_lo, overall_hi])

for dim in ["sex", "smoker", "time", "day", "size"]:
    for level in sorted(tips[dim].unique(), key=str):
        sub = tips.loc[tips[dim] == level, "tip_pct"]
        pt, lo, hi = bootstrap_ci(sub)
        rows.append([dim, str(level), len(sub), pt, lo, hi])

ci = pd.DataFrame(rows, columns=["维度", "层级", "样本量", "均值", "CI下界", "CI上界"])
ci["区间宽度pp"] = (ci["CI上界"] - ci["CI下界"]) * 100
for c in ["均值", "CI下界", "CI上界"]:
    ci[c] = ci[c] * 100
ci["覆盖全体均值"] = (ci["CI下界"] <= overall_pt * 100) & (ci["CI上界"] >= overall_pt * 100)

print(ci.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

print("\n" + "=" * 92)
print("精度诊断")
print("=" * 92)
wide = ci[(ci["区间宽度pp"] > 4) & (ci["维度"] != "全体")]
print(f"区间宽度 > 4 个百分点（估计极不稳定）的分组：{len(wide)} 个")
if len(wide) > 0:
    print(wide[["维度", "层级", "样本量", "均值", "区间宽度pp"]]
          .sort_values("区间宽度pp", ascending=False)
          .to_string(index=False, float_format=lambda x: f"{x:.2f}"))
    print("→ 这些分组不应单独作为决策依据。")

not_cover = ci[(~ci["覆盖全体均值"]) & (ci["维度"] != "全体")]
print(f"\n置信区间不包含全体均值（真正与整体不同）的分组：{len(not_cover)} 个")
if len(not_cover) > 0:
    print(not_cover[["维度", "层级", "样本量", "均值", "CI下界", "CI上界"]]
          .to_string(index=False, float_format=lambda x: f"{x:.2f}"))
else:
    print("→ 所有分组的区间都覆盖了全体均值，与步骤5 的检验结论一致：细分证据不足。")

fig, ax = plt.subplots(figsize=(11, 6))
plot_df = ci[ci["维度"].isin(["size", "day"])].copy()
plot_df["标签"] = plot_df["维度"] + "=" + plot_df["层级"]
y = np.arange(len(plot_df))
err_lo = plot_df["均值"] - plot_df["CI下界"]
err_hi = plot_df["CI上界"] - plot_df["均值"]
ax.errorbar(plot_df["均值"], y, xerr=[err_lo, err_hi], fmt="o",
            color="#2563eb", ecolor="#93c5fd", elinewidth=3, capsize=4, markersize=7)
ax.axvline(overall_pt * 100, color="#dc2626", linestyle="--", linewidth=1.6,
           label=f"全体均值 {overall_pt * 100:.2f}%")
ax.set_yticks(y)
ax.set_yticklabels(plot_df["标签"])
ax.set_xlabel("小费率 (%)")
ax.set_title("各分组小费率的 95% 自举置信区间（横线越长=越不可信）")
ax.legend()
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


## 步骤7：连续变量归因 —— 账单金额与小费率的真实关系

分类维度全部失效之后，回到唯一的连续变量：账单金额。这里要同时做三件事——分箱看单调性、手工最小二乘拟合斜率、以及对比 tip 绝对额与 tip 率的相反结论。最后一点是本项目最重要的商业洞察：**指标选错，结论就会反向**。手写 OLS（不调库）能让学员真正理解回归系数是怎么算出来的。


In [ ]:
print("=" * 92)
print("账单金额 → 小费率：分箱 + 手工 OLS")
print("=" * 92)

tips["bill_bin"] = pd.qcut(tips["total_bill"], q=5,
                           labels=["最低20%", "较低", "中等", "较高", "最高20%"])
binned = tips.groupby("bill_bin", observed=True).agg(
    样本量=("tip_pct", "size"),
    账单均值=("total_bill", "mean"),
    小费额均值=("tip", "mean"),
    小费率均值=("tip_pct", "mean"),
).reset_index()
binned["小费率均值"] = binned["小费率均值"] * 100
print(binned.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

first, last = binned.iloc[0], binned.iloc[-1]
print(f"\n从最低20% 到最高20%：")
print(f"  账单   {first['账单均值']:.2f} → {last['账单均值']:.2f} 元 "
      f"({last['账单均值'] / first['账单均值']:.2f} 倍)")
print(f"  小费额 {first['小费额均值']:.2f} → {last['小费额均值']:.2f} 元 "
      f"({last['小费额均值'] / first['小费额均值']:.2f} 倍)  ← 绝对额上升")
print(f"  小费率 {first['小费率均值']:.2f}% → {last['小费率均值']:.2f}%  "
      f"({last['小费率均值'] - first['小费率均值']:+.2f} pp)  ← 比率下降")
print("  两个指标给出方向相反的结论，这正是「选错指标就得错结论」的教科书案例。")

def ols_simple(x, y):
    """手工一元最小二乘：beta = Cov(x,y)/Var(x)"""
    x, y = np.asarray(x, float), np.asarray(y, float)
    xbar, ybar = x.mean(), y.mean()
    beta = ((x - xbar) * (y - ybar)).sum() / ((x - xbar) ** 2).sum()
    alpha = ybar - beta * xbar
    pred = alpha + beta * x
    resid = y - pred
    ss_res, ss_tot = (resid ** 2).sum(), ((y - ybar) ** 2).sum()
    r2 = 1 - ss_res / ss_tot
    se_beta = np.sqrt(ss_res / (len(x) - 2) / ((x - xbar) ** 2).sum())
    return {"alpha": alpha, "beta": beta, "r2": r2, "se_beta": se_beta,
            "t": beta / se_beta, "pred": pred, "resid": resid}

print("\n" + "-" * 92)
print("手工 OLS 拟合")
print("-" * 92)
m_rate = ols_simple(tips["total_bill"], tips["tip_pct"] * 100)
m_amt = ols_simple(tips["total_bill"], tips["tip"])
for label, m, unit in [("小费率(%)", m_rate, "pp"), ("小费额(元)", m_amt, "元")]:
    print(f"\n因变量 = {label}")
    print(f"  截距 alpha = {m['alpha']:+.4f}")
    print(f"  斜率 beta  = {m['beta']:+.5f} {unit} / 每元账单   (t = {m['t']:+.2f})")
    print(f"  R^2        = {m['r2']:.4f}   → 账单金额只能解释 {m['r2'] * 100:.1f}% 的变异")
    print(f"  10 元账单增量的影响: {m['beta'] * 10:+.3f} {unit}")

print(f"\n相关系数 corr(total_bill, tip)     = {tips['total_bill'].corr(tips['tip']):+.4f}")
print(f"相关系数 corr(total_bill, tip_pct) = {tips['total_bill'].corr(tips['tip_pct']):+.4f}")
print(f"人均账单 corr(bill_per_person, tip_pct) = "
      f"{tips['bill_per_person'].corr(tips['tip_pct']):+.4f}")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
axes[0].scatter(tips["total_bill"], tips["tip"], s=28, alpha=0.55, color="#2563eb")
order = np.argsort(tips["total_bill"].values)
axes[0].plot(tips["total_bill"].values[order], m_amt["pred"][order], color="#dc2626", linewidth=2)
axes[0].set_xlabel("账单金额 (元)"); axes[0].set_ylabel("小费额 (元)")
axes[0].set_title(f"小费额随账单上升 (R²={m_amt['r2']:.3f})")

axes[1].scatter(tips["total_bill"], tips["tip_pct"] * 100, s=28, alpha=0.55, color="#059669")
axes[1].plot(tips["total_bill"].values[order], m_rate["pred"][order], color="#dc2626", linewidth=2)
axes[1].axhline(tips["tip_pct"].mean() * 100, color="#6b7280", linestyle=":", linewidth=1.4)
axes[1].set_xlabel("账单金额 (元)"); axes[1].set_ylabel("小费率 (%)")
axes[1].set_title(f"小费率随账单下降 (R²={m_rate['r2']:.3f})")

axes[2].bar(binned["bill_bin"].astype(str), binned["小费率均值"], color="#7c3aed", alpha=0.85)
axes[2].axhline(tips["tip_pct"].mean() * 100, color="#dc2626", linestyle="--", linewidth=1.5)
axes[2].set_ylabel("小费率 (%)"); axes[2].set_title("分箱后的小费率单调性")
axes[2].tick_params(axis="x", rotation=20)
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 步骤8：结论与行动 —— 一份诚实的分析报告

把七步证据串成结论。这个项目的价值有一半来自**负面结论**：大部分人群细分差异经不起检验。敢于报告「证据不足」，并明确指出需要多少样本才能得出结论，是分析师专业性的核心体现——它避免了团队基于噪声去改排班、改服务话术。


In [ ]:
print("=" * 92)
print("分析报告：什么因素真正影响小费率")
print("=" * 92)

print("\n【一、事实层】")
print(f"  样本 {len(tips)} 单，平均小费率 {tips['tip_pct'].mean() * 100:.2f}%，"
      f"中位数 {tips['tip_pct'].median() * 100:.2f}%（右偏分布）")
print(f"  95% 自举区间 [{overall_lo * 100:.2f}%, {overall_hi * 100:.2f}%]")

print("\n【二、被证伪的假设】")
for _, r in res.iterrows():
    if r["显著"] == "否":
        print(f"  x {r['对比']}: 差异 {r['差异pp']:+.2f}pp, p={r['p值']:.3f}, "
              f"效应量{r['效应量']} → 无法与随机波动区分")

print("\n【三、成立的结论】")
print(f"  v 小费率随账单金额单调下降：每增加 10 元账单，小费率降低 "
      f"{abs(m_rate['beta'] * 10):.3f} 个百分点 (t={m_rate['t']:.2f})")
print(f"  v 但小费绝对额随账单上升：每增加 10 元账单，小费增加 {m_amt['beta'] * 10:.3f} 元")
print(f"  v 账单金额仅解释小费率 {m_rate['r2'] * 100:.1f}% 的变异 → 主要驱动因素不在本数据集内")
if reversals:
    print(f"  v 发现 {len(reversals)} 处符号反转，聚合结论不可直接外推到分层")
print("  v 午餐与工作日完全重合，该维度不可做因果解读")

print("\n" + "=" * 92)
print("行动建议")
print("=" * 92)
for i, (act, why) in enumerate([
    ("以「小费总额」而非「小费率」作为北极星指标", "率随账单下降但额上升，选错指标会得到反向结论"),
    ("优先做提升客单价的动作（套餐、加菜推荐）", "小费额与账单强正相关，是唯一被数据证实的杠杆"),
    ("暂停一切基于性别/吸烟/餐段的差异化服务策略", "这些差异均未通过置换检验，属于噪声"),
    ("补采服务员ID、等待时长、支付方式等字段", "现有变量只解释了不到20%的变异，主因缺失"),
    ("将样本量扩到千级后重做分层检验", "当前每层样本不足，区间宽度普遍超过4个百分点"),
], 1):
    print(f"{i}. {act}\n   依据: {why}")

need_n = int(np.ceil(2 * (tips['tip_pct'].std() / 0.01) ** 2 * (1.96 + 0.84) ** 2 / 2))
print(f"\n样本量估算：若要在 alpha=0.05、power=0.8 下检出 1 个百分点的组间差异，")
print(f"每组约需 {need_n} 单（当前最大分组仅 {tips['sex'].value_counts().max()} 单）。")

print("\n分析局限：单店快照数据，无时间维度，无法排除季节性与服务员个体效应。")


## 结论与表达

- 全体平均小费率约 15%，但分布右偏，少数小额账单产生了极高的小费率——用中位数与自举区间比单一均值更可靠。
- 性别、吸烟区、餐段、周末等人群细分差异全部未通过 5000 次置换检验，Cohen's d 均在「可忽略」到「小」区间，属于噪声而非规律。
- 唯一稳健的规律是账单金额：小费率随账单单调下降，而小费绝对额随账单上升——同一份数据用不同指标会得出方向相反的结论。
- 账单金额只解释了小费率约 5% 的变异，说明真正的驱动因素（服务质量、等待时长、服务员个人）不在当前数据集内，结论必须停在「证据不足」而不是编造解释。


## 项目验收清单

- 能手写置换检验并解释为什么它不需要正态性假设
- 能说明自举置信区间的宽度反映了什么，以及区间覆盖全体均值意味着什么
- 能在交叉分层中识别符号反转，并解释辛普森悖论的成因
- 能说明为什么「小费率下降」和「小费额上升」可以同时成立，以及该如何选择业务指标

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 本章小结

餐厅想提高小费收入，但不知道该从哪里下手：是抓大单、调班次、还是改服务对象？本项目用真实的 244 条餐厅账单记录，做分层归因和统计推断，区分「看起来有差异」和「真的有差异」。


### 你已经完成

- 掌握派生指标的设计：为什么要用小费率而不是小费绝对值做分析
- 掌握分层分析（stratification）：逐维度拆解 + 样本量校验
- 识别辛普森悖论：交叉分层后结论反转的成因
- 在不依赖 scipy 的前提下，用置换检验判断组间差异的统计显著性
- 用自举法（bootstrap）给出效应量的置信区间，而不是只报一个点估计


### 项目流程速查

| 阶段 | 交付内容 |
| --- | --- |
| 步骤 1 | 构造小费率指标，并说明为什么它比小费绝对值更适合做跨账单比较 |
| 步骤 2 | 对性别、吸烟区、星期、餐段、人数逐一做单因素分层，记录均值差与样本量 |
| 步骤 3 | 做「餐段 × 星期」交叉表，找出单因素结论在交叉后被推翻的维度 |
| 步骤 4 | 对最大的那个组间差异做置换检验，报告 p 值 |
| 步骤 5 | 用自举法给出该差异的 95% 置信区间，判断它是否跨过 0 |
| 步骤 6 | 汇总证据，给出可执行的行动建议并标注置信度 |


### 质量与结论提醒

- 确认无缺失值与非法值（账单金额、小费必须为正）
- 检查重复行：完全相同的账单是否真实存在
- 逐维度统计各分组样本量，标记 n < 20 的格子为「不可结论区」
- 全体平均小费率约 15%，但分布右偏，少数小额账单产生了极高的小费率——用中位数与自举区间比单一均值更可靠。
- 性别、吸烟区、餐段、周末等人群细分差异全部未通过 5000 次置换检验，Cohen's d 均在「可忽略」到「小」区间，属于噪声而非规律。
- 唯一稳健的规律是账单金额：小费率随账单单调下降，而小费绝对额随账单上升——同一份数据用不同指标会得出方向相反的结论。
- 账单金额只解释了小费率约 5% 的变异，说明真正的驱动因素（服务质量、等待时长、服务员个人）不在当前数据集内，结论必须停在「证据不足」而不是编造解释。


### 项目交付检查

- [ ] 能手写置换检验并解释为什么它不需要正态性假设
- [ ] 能说明自举置信区间的宽度反映了什么，以及区间覆盖全体均值意味着什么
- [ ] 能在交叉分层中识别符号反转，并解释辛普森悖论的成因
- [ ] 能说明为什么「小费率下降」和「小费额上升」可以同时成立，以及该如何选择业务指标
